# Deep Convolutional Models

Discover some powerful practical tricks and methods used in deep CNNs, straight from the research papers, then apply transfer learning to your own deep CNN.

**Learning Objectives**
* Implement the basic building blocks of ResNets in a deep neural network using Keras
* Train a state-of-the-art neural network for image classification
* Implement a skip connection in your network
* Create a dataset from a directory
* Preprocess and augment data using the Keras Sequential API
* Adapt a pretrained model to new data and train a classifier using the Functional API and MobileNet
* Fine-tine a classifier's final layers to improve accuracy

---

## Table of Contents


---

## Why look at case studies?

This section serves as an introduction to the importance of studying specific **CNN Case Studies** to build intuition for designing deep learning architectures.

Here is key points why we should get ourselves familiar with CNN case studies:
* **Intuition Building:** Just as programmers learn by reading others' code, machine learning practitioners gain intuition by seeing how researchers combined building blocks (Convolutions, Pooling, Fully Connected layers).
* **Transferability:** A neural network architecture that performs well on one computer vision task (e.g., recognizing animals) often performs exceptionally well on unrelated tasks (e.g., self-driving car perception).
* **Foundation for Research:** Understanding these classic architectures provides the vocabulary and structural knowledge required to read and comprehend professional computer vision research papers.
* **Cross-Disciplinary Value:** Concepts born in computer vision (like Residual connections) are frequently "cross-fertilized" into other AI disciplines.

### Key Architectures to be Covered

| Category | Models | Significance |
| --- | --- | --- |
| **Classic Networks** | **LeNet-5**, **AlexNet**, **VGG** | The foundational models that established the standard "Conv-Pool-FC" pattern. |
| **Residual Networks** | **ResNet** | Introduced "skip connections" to train extremely deep networks (up to 152 layers) effectively. |
| **Inception Networks** | **Inception** | Focuses on computational efficiency and multi-scale processing within a single layer. |

### 🔢 Mathematical Intuition: The Depth Problem

The transcript mentions that ResNet allows for $152$ layers. Normally, as the number of layers $L$ increases, it becomes harder to train due to the vanishing gradient problem. In a standard network, the gradient of the loss $\mathcal{L}$ with respect to the weights of the first layer $W^{[1]}$ is:

$$\frac{\partial \mathcal{L}}{\partial W^{[1]}} = \frac{\partial \mathcal{L}}{\partial a^{[L]}} \cdot \frac{\partial a^{[L]}}{\partial a^{[L-1]}} \dots \frac{\partial a^{[2]}}{\partial a^{[1]}} \cdot \frac{\partial a^{[1]}}{\partial W^{[1]}}$$

If the derivatives are small ($<1$), multiplying them over $152$ layers causes the gradient to shrink to zero. ResNet uses "tricks" (skip connections) to create a "highway" for the gradient, allowing deep networks to converge.

---

## Classic Networks

This section details three "classic" CNN architectures — **LeNet-5**, **AlexNet**, and **VGG-16** — which established the foundational design patterns used in modern computer vision.

### 1. LeNet-5 (1998)

The first successful application of CNNs, designed by Yann LeCun for handwritten digit recognition ($0-9$).

* **Input:** $32 \times 32 \times 1$ grayscale images.
* **Architecture:** Follows a repeating pattern of $CONV \rightarrow POOL \rightarrow CONV \rightarrow POOL \rightarrow FC \rightarrow FC \rightarrow Output$.
* **Key Traits:** Used **Average Pooling** and **Sigmoid/Tanh** activations (standard for the time).
    * Small by modern standards with only **60,000 parameters**.
* **Trend:** As you go deeper, the height/width ($n_H, n_W$) decrease while the number of channels ($n_C$) increases ($1 \rightarrow 6 \rightarrow 16$).
* **Reference:** LeCun, Y., Bottou, L., Bengio, Y., and Haffner, P. (1998). "Gradient-based learning applied to document recognition." Proceedings of the IEEE, 86(11), 2278-2324. — Section II provides the detailed description of the architecture.

<div style="display: flex; justify-content: center;">
    <img src="images/LeNet-5.png" width="700px">
</div>

### 2. AlexNet (2012)

The model that triggered the modern Deep Learning revolution by winning the ImageNet competition.

* **Input:** $227 \times 227 \times 3$ RGB images.
* **Architecture:** Much deeper and larger than LeNet, with **60 million parameters**.
* **Innovations:**
    * Introduced the **ReLU** activation function, which significantly sped up training.
    * Used **Max Pooling** instead of Average Pooling.
    * Utilized **Multiple GPUs** (splitting the model across two cards).
    * Applied **Dropout** and **Data Augmentation** (though not detailed in this section).
* **Legacy:** Convinced the vision community that Deep Learning was superior to traditional hand-coded features.
* **Reference:** Krizhevsky, A., Sutskever, I., and Hinton, G. E. (2012). "ImageNet classification with deep convolutional neural networks." Advances in Neural Information Processing Systems (NeurIPS), 25, 1097-1105. — This is widely considered the "easiest" of the three to read for beginners.

<div style="display: flex; justify-content: center;">
    <img src='images/AlexNet.png' width='700px'>
</div>

### 3. VGG-16 (2014)

Designed for simplicity and uniformity, VGG-16 replaced complex hyperparameters with a standardized "building block" approach.

* **Standardized Structure:** 
    * **CONV:** Always $3 \times 3$ filters, stride 1, and `same` padding.
    * **POOL:** Always $2 \times 2$ Max Pooling with stride 2.
* **The "Doubling" Pattern:** Every time a pooling layer halves the height and width, the number of filters in the next convolution layer **doubles** ($64 \rightarrow 128 \rightarrow 256 \rightarrow 512$).
* **Parameters:** Very heavy, containing roughly **138 million parameters**.
* **VGG-16 vs. VGG-19:** VGG-19 is a larger version, but VGG-16 is often preferred because it performs almost as well with fewer parameters.
* **Reference:** Simonyan, K. and Zisserman, A. (2014). "Very deep convolutional networks for large-scale image recognition." arXiv preprint arXiv:1409.1556. (Later published at ICLR 2015). — Look for the tables comparing VGG-16 and VGG-19 to see how increasing depth impacts performance.

<div style='dipslay: flex: justify-content: center'>
    <img src='images/VGG-16.png' width='750px'>
</div>

### Summary Comparison

| Metric | LeNet-5 | AlexNet | VGG-16 |
| --- | --- | --- | --- |
| **Year** | 1998 | 2012 | 2014 |
| **Parameters** | ~60K | ~60M | ~138M |
| **Activation** | Sigmoid / Tanh | ReLU | ReLU |
| **Pooling** | Average | Max | Max |
| **Key Strength** | First real CNN | Scale & GPU usage | Simplicity & Uniformity |

### Recurring Design Patterns

Across all three models, a consistent "Rule of Thumb" emerged:

1. **Spatial Shrinkage:** Height and width decrease as you go deeper into the network.
2. **Channel Expansion:** The number of channels (depth of the volume) increases as you go deeper.
3. **The Stack:** Most networks consist of several Conv-Pool blocks followed by several Fully Connected (FC) layers.

---

## Residual Networks (ResNets)

This section explains the concept of **Residual Networks (ResNets)**, focusing on how **skip connections** (or shortcuts) solve the problem of training extremely deep neural networks.

### Key Takeaways

* **The Vanishing/Exploding Gradient Problem:** In standard "Plain" networks, as the number of layers increases, gradients can become extremely small or large, making it nearly impossible for optimization algorithms (like Gradient Descent) to train early layers effectively.
* **The Residual Block:** The fundamental unit of a ResNet. Instead of information flowing only through a linear path, a "shortcut" is created that passes the activation $a^{[l]}$ directly to a deeper layer, bypassing one or more intermediate layers.
* **The Shortcut/Skip Connection:** Information from $a^{[l]}$ is added to the linear output $z^{[l+2]}$ *before* the final ReLU non-linearity is applied.

<div style='display: flex; justify-content: center'>
    <img src='images/resblock.png' width='750px'>
</div>

* **The ResNet Advantage:** 
    * **In Plain Networks:** Empirically, training error actually **increases** once a plain network gets too deep because optimization becomes too difficult.
    * **In ResNets:** Even with over **100 layers**, the training error continues to decrease or plateaus, allowing for much deeper architectures without performance degradation.

<div style='display: flex; justify-content: center'>
    <img src='images/resneterror.png' width='750px'>
</div>

* **Reference article:** The concept was introduced by Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun [here](https://www.cv-foundation.org/openaccess/content_cvpr_2016/papers/He_Deep_Residual_Learning_CVPR_2016_paper.pdf).

### Structural Comparison: Plain vs. Residual

In a **Plain Network**, the path is strictly linear:

<div style='display: flex; justify-content: center'>
    <img src='images/plainnet.png' width='750px'>
</div>

In a **ResNet**, skip connections create a "highway" for information:

<div style='display: flex; justify-content: center'>
    <img src='images/resnet.png' width='750px'>
</div>

### Mathematical Intuition: The Residual Step

In a standard network, the activation $a^{[l+2]}$ is calculated as:

$$a^{[l+2]} = g(z^{[l+2]})$$

In a **Residual Block**, the activation is modified to include the "shortcut" from two layers prior:

$$a^{[l+2]} = g(z^{[l+2]} + a^{[l]})$$

This addition makes it very easy for the network to learn the **identity function** ($a^{[l+2]}=a^{[l]}$). If the intermediate layers learn to set their weights to zero, the information simply passes through unchanged, ensuring that a deeper network performs *at least as well* as a shallower one.

### Performance in Practice

The primary breakthrough of ResNet is observed in the training error vs. depth graph:

| Network Type | 20 Layers | 50 Layers | 100+ Layers |
| --- | --- | --- | --- |
| **Plain Network** | Good | Worse (Optimization fails) | Very Poor |
| **ResNet** | Good | Better | Excellent |